# 07 · Balón parado

**Último bloque del Plan Maestro (parte de la pregunta 2, ideas tácticas del entrenador):** patrones ofensivos de América en córners, tiros libres y saques de banda (zonas de remate, tipo de envío), y organización defensiva ante el balón parado rival (zonas de vulnerabilidad).

Se analizan 3 tipos de jugada a balón parado, cada uno con dos vistas:
- **Ofensiva:** dónde remata América tras su propio saque, y qué tipo de envío usa.
- **Defensiva:** dónde remata el RIVAL tras su propio saque, dentro de los mismos partidos — esto mide la vulnerabilidad de América ante el balón parado en contra.

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework_analitico import (
    normalizar_direccion_ataque,
    resumen_balon_parado,
    remates_balon_parado,
    perfil_envios_balon_parado,
    dibujar_zona_porteria,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)
print(f"Partidos: {events_norm['match_id'].nunique()}")

## 1. Efectividad ofensiva por tipo de balón parado

Cuántos se sacaron, qué % termina en remate propio, y el xG generado.

In [ ]:
tipos = ["corner", "tiro_libre", "saque_banda"]
resumen = pd.DataFrame([resumen_balon_parado(events_norm, equipo=EQUIPO, tipo=t) for t in tipos])
resumen

## 2. Córners — zona de remate (propio) y tipo de envío

In [ ]:
remates_corner_favor = remates_balon_parado(events_norm, equipo=EQUIPO, tipo="corner", propio=True)
print(f"Remates tras córner (propio): {len(remates_corner_favor)}")

envios_corner = perfil_envios_balon_parado(events_norm, equipo=EQUIPO, tipo="corner")
print(f"Córners botados: {len(envios_corner)}")
print()
print(envios_corner["pass_technique"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(remates_corner_favor["x"], remates_corner_favor["y"], alpha=0.6, color="#FFD700", edgecolor="black", zorder=2)
dibujar_zona_porteria(axes[0], lado="ataque", color="#444444")
axes[0].set_xlim(60, 120)
axes[0].set_ylim(0, 80)
axes[0].invert_yaxis()
axes[0].set_title(f"Zona de remate tras córner propio (n={len(remates_corner_favor)})")

axes[1].scatter(envios_corner["end_x"], envios_corner["end_y"], alpha=0.4, color="#8B0000", zorder=2)
dibujar_zona_porteria(axes[1], lado="ataque", color="#444444")
axes[1].set_xlim(60, 120)
axes[1].set_ylim(0, 80)
axes[1].invert_yaxis()
axes[1].set_title(f"Punto de llegada de los córners botados (n={len(envios_corner)})")

plt.tight_layout()
plt.savefig("../outputs/figures/corners_ofensivo.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Tiros libres — zona de remate (propio) y tipo de envío

In [ ]:
remates_tl_favor = remates_balon_parado(events_norm, equipo=EQUIPO, tipo="tiro_libre", propio=True)
print(f"Remates tras tiro libre (propio): {len(remates_tl_favor)}")

envios_tl = perfil_envios_balon_parado(events_norm, equipo=EQUIPO, tipo="tiro_libre")
print(f"Tiros libres botados: {len(envios_tl)}")
print()
print(envios_tl["pass_technique"].value_counts())
print()
# Tiros libres DIRECTOS (remate directo a gol, sin pase intermedio)
directos = events_norm[
    (events_norm.team == EQUIPO) & (events_norm.type == "Shot") & (events_norm.shot_type == "Free Kick")
]
print(f"Tiros libres directos a portería: {len(directos)}")

## 4. Vulnerabilidad defensiva: ¿dónde remata el rival tras SU balón parado?

Misma vista que las secciones 2-3, pero del lado defensivo — dónde genera peligro el rival tras sus propios córners y tiros libres, dentro de los partidos de América.

In [ ]:
remates_corner_contra = remates_balon_parado(events_norm, equipo=EQUIPO, tipo="corner", propio=False)
remates_tl_contra = remates_balon_parado(events_norm, equipo=EQUIPO, tipo="tiro_libre", propio=False)

print(f"Remates del RIVAL tras córner: {len(remates_corner_contra)}")
print(f"Remates del RIVAL tras tiro libre: {len(remates_tl_contra)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(remates_corner_contra["x"], remates_corner_contra["y"], alpha=0.6, color="#8B0000", edgecolor="black", zorder=2)
dibujar_zona_porteria(axes[0], lado="defensa", color="#444444")
axes[0].set_xlim(0, 60)
axes[0].set_ylim(0, 80)
axes[0].invert_yaxis()
axes[0].set_title(f"Remates del rival tras SU córner (n={len(remates_corner_contra)})")

axes[1].scatter(remates_tl_contra["x"], remates_tl_contra["y"], alpha=0.6, color="#8B0000", edgecolor="black", zorder=2)
dibujar_zona_porteria(axes[1], lado="defensa", color="#444444")
axes[1].set_xlim(0, 60)
axes[1].set_ylim(0, 80)
axes[1].invert_yaxis()
axes[1].set_title(f"Remates del rival tras SU tiro libre (n={len(remates_tl_contra)})")

plt.tight_layout()
plt.savefig("../outputs/figures/balon_parado_defensivo.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Comparación a favor vs. en contra: xG generado vs. xG concedido

In [ ]:
def xg_total(df):
    return round(df["shot_statsbomb_xg"].sum(), 2) if "shot_statsbomb_xg" in df.columns and len(df) > 0 else 0.0

comparacion_xg = pd.DataFrame({
    "tipo": ["corner", "tiro_libre"],
    "xg_a_favor": [xg_total(remates_corner_favor), xg_total(remates_tl_favor)],
    "xg_en_contra": [xg_total(remates_corner_contra), xg_total(remates_tl_contra)],
})
comparacion_xg["diferencia"] = comparacion_xg["xg_a_favor"] - comparacion_xg["xg_en_contra"]
comparacion_xg

## 6. Síntesis: patrones de balón parado

*(completar una vez corridas las celdas anteriores con los resultados reales — con esto se cierran los 7 bloques del Plan Maestro)*